# nb_workspace_monitoring — querying the Monitoring Eventhouse with KQL
**What this is:** Fabric **Workspace Monitoring** provisions a read-only Eventhouse KQL database in
the workspace that centrally collects logs/metrics from Fabric items. Enable it under
**Workspace settings → Monitoring → Log workspace activity**; data arrives with ~10–15 min latency,
default retention 30 days, Contributor+ role required to query. Billing rides the capacity, but
monitoring ingestion/queries keep working even when the capacity is throttled — which is exactly
why it's a good place to watch throttling from.

**What lands there today:** `ItemJobEvents` (job performance/trends for Fabric items — the all-up
table this notebook leans on), Power BI `SemanticModelLogs`, the Eventhouse family
(`EventhouseQueryLogs`, `EventhouseCommandLogs`, `EventhouseDataOperationLogs`,
`EventhouseIngestionResultLogs`, `EventhouseMetrics`), GraphQL and mirrored-database logs.
**Spark application logs are NOT yet a native workspace-monitoring table** — for Spark internals
telemetry the supported pattern is the **diagnostic emitter → Event Hubs → Eventstream → Eventhouse**
(section 5), landing Spark events in the *same* KQL estate so one query surface covers everything.

**Platform pattern:** discovery first (`.show tables` — table sets evolve; if a documented table is
missing, toggle Log workspace activity off/on to refresh the schema), parameterized KQL as data,
results joined back to the run-log via `applicationId`.

In [1]:
NOTEBOOK_NAME = "nb_workspace_monitoring"
# Monitoring Eventhouse "Query URI" - copy from the monitoring KQL database's details page
KUSTO_QUERY_URI = "https://<eventhouse>.z<region>.kusto.fabric.microsoft.com"
KUSTO_DATABASE  = "Monitoring Eventhouse"
LOOKBACK        = "1d"          # KQL timespan literal: 1h, 1d, 7d ...
DEMO_MODE       = None          # None = auto (demo locally, live in Fabric); or force True/False

In [2]:
import json, os
from datetime import datetime, timedelta, timezone

def detect_fabric():
    try:
        import notebookutils  # noqa
        return True
    except ImportError:
        return False

IN_FABRIC = detect_fabric()
DEMO = DEMO_MODE if DEMO_MODE is not None else (not IN_FABRIC)
print(f"fabric={IN_FABRIC} demo_mode={DEMO}")

fabric=False demo_mode=True


## 1 — Connect and run KQL
Two supported client paths from a notebook; both authenticate with the notebook identity's Entra
token (no secrets in code). Path A (pure Python, works in Python notebooks too) is the default.
In demo mode the same `run_kql()` returns simulated frames so every analysis cell below executes
and shows the intended output shape — clearly labelled, no pretend connectivity.

In [3]:
import pandas as pd

KQL = {}   # the query library - populated in section 2; queries are data, not inline strings

def _demo_frames():
    now = datetime.now(timezone.utc)
    import random; random.seed(7)
    items = [("nb_ingestion_generic","Notebook"),("nb_data_quality","Notebook"),
             ("pl_master_load","DataPipeline"),("nb_lakehouse_maintenance","Notebook"),
             ("sjd_gold_build","SparkJobDefinition")]
    rows = []
    for i in range(300):
        name, kind = random.choice(items)
        dur = random.expovariate(1/180) + 20
        ok  = random.random() > (0.18 if name == "pl_master_load" else 0.04)
        st  = now - timedelta(minutes=random.uniform(0, 24*60))
        rows.append({"Timestamp": st, "ItemName": name, "ItemKind": kind,
                     "JobStatus": "Completed" if ok else "Failed",
                     "DurationSec": round(dur,1),
                     "ExecutingUser": random.choice(["naresh@corp","svc-etl@corp"])})
    return {"ItemJobEvents": pd.DataFrame(rows)}

_DEMO_DATA = _demo_frames() if DEMO else None

def run_kql(query: str) -> pd.DataFrame:
    """Live: azure-kusto-data with the notebook's Entra token. Demo: simulated equivalents."""
    if DEMO:
        return _demo_kql(query)
    from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
    import notebookutils
    token = notebookutils.credentials.getToken(KUSTO_QUERY_URI)
    kcsb = KustoConnectionStringBuilder.with_token_provider(KUSTO_QUERY_URI, lambda: token)
    client = KustoClient(kcsb)
    result = client.execute(KUSTO_DATABASE, query)
    return pd.DataFrame([r.to_dict() for r in result.primary_results[0]])

# Alternative (Spark notebooks): the Kusto Spark connector -
SPARK_CONNECTOR_SNIPPET = """
df = (spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
      .option("kustoCluster", KUSTO_QUERY_URI)
      .option("kustoDatabase", KUSTO_DATABASE)
      .option("accessToken", notebookutils.credentials.getToken(KUSTO_QUERY_URI))
      .option("kustoQuery", KQL["job_failures"]).load())
"""
print("run_kql ready |", "DEMO (simulated frames)" if DEMO else "LIVE (azure-kusto-data)")

run_kql ready | DEMO (simulated frames)


## 2 — The KQL query library
Queries are **data** — stored here (in production: a `monitoring_queries` table or Queryset),
executed by the generic runner. Each answers one operational question. Microsoft publishes more in
the `fabric-samples` GitHub repo under `workspace-monitoring`.

In [4]:
KQL["discovery"] = ".show tables"

KQL["job_failures"] = f"""
ItemJobEvents
| where Timestamp > ago({LOOKBACK})
| where JobStatus == "Failed"
| summarize failures = count(), lastFailure = max(Timestamp) by ItemName, ItemKind
| order by failures desc
"""

KQL["failure_rate_trend"] = f"""
ItemJobEvents
| where Timestamp > ago(7d)
| summarize total = count(), failed = countif(JobStatus == "Failed") by bin(Timestamp, 1h), ItemKind
| extend failure_rate = round(100.0 * failed / total, 1)
| order by Timestamp asc
"""

KQL["slowest_jobs"] = f"""
ItemJobEvents
| where Timestamp > ago({LOOKBACK}) and JobStatus == "Completed"
| top 20 by DurationSec desc
| project Timestamp, ItemName, ItemKind, DurationSec, ExecutingUser
"""

KQL["duration_regression"] = f"""
ItemJobEvents
| where Timestamp > ago(14d) and JobStatus == "Completed"
| summarize p50 = percentile(DurationSec, 50), p95 = percentile(DurationSec, 95), runs = count()
    by ItemName, bin(Timestamp, 1d)
| order by ItemName asc, Timestamp asc
"""

KQL["eventhouse_slow_queries"] = f"""
EventhouseQueryLogs
| where Timestamp > ago({LOOKBACK})
| top 20 by DurationMs desc
| project Timestamp, DatabaseName, Text, DurationMs, CacheColdHitsBytes, User
"""

KQL["ingestion_failures"] = f"""
EventhouseIngestionResultLogs
| where Timestamp > ago({LOOKBACK}) and ResultCode != "Success"
| summarize count() by ResultCode, Database = DatabaseName
"""
for k in KQL: print(" -", k)

 - discovery
 - job_failures
 - failure_rate_trend
 - slowest_jobs
 - duration_regression
 - eventhouse_slow_queries
 - ingestion_failures


In [5]:
def _demo_kql(query: str) -> pd.DataFrame:
    """Demo-mode equivalents over the simulated ItemJobEvents (pandas re-implementation of the
    KQL semantics for the two queries the analysis below uses; others return an empty frame with
    a note - the KQL text itself is the deliverable)."""
    df = _DEMO_DATA["ItemJobEvents"]
    if "JobStatus == \"Failed\"" in query and "summarize failures" in query:
        f = df[df.JobStatus == "Failed"]
        out = (f.groupby(["ItemName","ItemKind"])
                 .agg(failures=("JobStatus","count"), lastFailure=("Timestamp","max"))
                 .reset_index().sort_values("failures", ascending=False))
        return out
    if "top 20 by DurationSec" in query:
        return (df[df.JobStatus=="Completed"].nlargest(20, "DurationSec")
                [["Timestamp","ItemName","ItemKind","DurationSec","ExecutingUser"]])
    if query.strip() == ".show tables":
        return pd.DataFrame({"TableName": ["ItemJobEvents","SemanticModelLogs",
            "EventhouseQueryLogs","EventhouseCommandLogs","EventhouseDataOperationLogs",
            "EventhouseIngestionResultLogs","EventhouseMetrics","GraphQLLogs"]})
    return pd.DataFrame({"note": [f"demo mode - run against the live Monitoring Eventhouse"]})

## 3 — Analysis: the three questions to ask every morning

In [6]:
print("=== Available tables (discovery first - schemas evolve) ===")
print(run_kql(KQL["discovery"]).to_string(index=False))

=== Available tables (discovery first - schemas evolve) ===
                    TableName
                ItemJobEvents
            SemanticModelLogs
          EventhouseQueryLogs
        EventhouseCommandLogs
  EventhouseDataOperationLogs
EventhouseIngestionResultLogs
            EventhouseMetrics
                  GraphQLLogs


In [7]:
fails = run_kql(KQL["job_failures"])
print(f"=== Failures in the last {LOOKBACK} ===")
print(fails.to_string(index=False) if len(fails) else "none")
if len(fails):
    worst = fails.iloc[0]
    print(f"\n-> triage: open '{worst.ItemName}' in the Monitoring hub; if it is a notebook, "
          f"cross-reference etl_run_log on run window, then jump to the Spark application by app_id.")
assert isinstance(fails, pd.DataFrame)

=== Failures in the last 1d ===
                ItemName           ItemKind  failures                      lastFailure
          pl_master_load       DataPipeline         8 2026-08-02 12:02:54.444132+00:00
         nb_data_quality           Notebook         5 2026-08-02 11:59:12.042240+00:00
nb_lakehouse_maintenance           Notebook         5 2026-08-02 08:30:59.615920+00:00
    nb_ingestion_generic           Notebook         2 2026-08-01 19:26:44.684805+00:00
          sjd_gold_build SparkJobDefinition         2 2026-08-02 13:24:02.127342+00:00

-> triage: open 'pl_master_load' in the Monitoring hub; if it is a notebook, cross-reference etl_run_log on run window, then jump to the Spark application by app_id.


In [8]:
slow = run_kql(KQL["slowest_jobs"])
print("=== Slowest completed jobs ===")
print(slow.head(8).to_string(index=False))
# The cross-reference that makes this actionable: our run logs carry the Spark applicationId,
# so a slow ItemJobEvents row joins to the exact Spark application for Sec-19-style drill-down.
print("\nJoin key pattern: ItemJobEvents(ItemName, window) <-> etl_run_log(run_id, app_id) "
      "<-> Monitoring hub application detail.")

=== Slowest completed jobs ===
                       Timestamp                 ItemName     ItemKind  DurationSec ExecutingUser
2026-08-02 04:00:04.352049+00:00           pl_master_load DataPipeline        912.0  svc-etl@corp
2026-08-01 17:22:32.313532+00:00 nb_lakehouse_maintenance     Notebook        839.1   naresh@corp
2026-08-02 11:26:43.369533+00:00     nb_ingestion_generic     Notebook        813.5   naresh@corp
2026-08-01 14:08:33.575135+00:00     nb_ingestion_generic     Notebook        776.9   naresh@corp
2026-08-01 17:29:18.548624+00:00          nb_data_quality     Notebook        747.6   naresh@corp
2026-08-02 13:02:35.720979+00:00           pl_master_load DataPipeline        708.0  svc-etl@corp
2026-08-01 22:59:37.969336+00:00          nb_data_quality     Notebook        682.3  svc-etl@corp
2026-08-02 12:13:09.083169+00:00     nb_ingestion_generic     Notebook        677.8   naresh@corp

Join key pattern: ItemJobEvents(ItemName, window) <-> etl_run_log(run_id, app_id) <-> 

## 4 — Alerting: from queries to signals
Point **Activator** at the monitoring Eventhouse (or a KQL Queryset) and alert on the queries above —
failure count > 0 for tier-1 items, p95 duration doubling week-over-week, ingestion failures — instead
of reading dashboards. Note the asymmetry: monitoring *ingestion and KQL queries* keep working during
capacity throttling, but Power BI reports and Activator alerts built on the database respect capacity
state — so for throttling early-warning specifically, pair with the Capacity Metrics App (Sec 18 of
the internals doc). Microsoft also ships real-time dashboard **templates** that connect straight to
the monitoring Eventhouse for a zero-build starting point.

## 5 — Getting SPARK logs into the same Eventhouse
Workspace monitoring doesn't ingest Spark application logs natively yet (coverage is expanding
workload by workload). The supported route today:

1. **Fabric Apache Spark diagnostic emitter** on the Environment, targeting **Azure Event Hubs**
   (`spark.synapse.diagnostic.emitters: EH`, `...EH.type: AzureEventHub`, categories
   `Log,EventLog,Metrics`, secret via Key Vault).
2. An **Eventstream** item with the Event Hub as source and the **Eventhouse as destination** —
   Spark driver/executor logs, Spark events and metrics land as KQL tables next to `ItemJobEvents`.
3. Query both with one KQL surface: correlate a `Failed` ItemJobEvents row with the Spark
   executor stderr that caused it, in one query, in one database.

Microsoft's **"Spark Monitoring and Optimization" Fabric jumpstart** deploys exactly this stack
(Eventstream + Eventhouse + real-time dashboard + an analyzer notebook) in one shot if you'd rather
not assemble it by hand — worth evaluating before building bespoke.

In [9]:
EMITTER_TO_EVENTHOUSE = """
# Environment -> Spark properties (session-start scope):
spark.synapse.diagnostic.emitters: EH
spark.synapse.diagnostic.emitter.EH.type: AzureEventHub
spark.synapse.diagnostic.emitter.EH.categories: Log,EventLog,Metrics
spark.synapse.diagnostic.emitter.EH.secret.keyVault: https://<kv>.vault.azure.net/
spark.synapse.diagnostic.emitter.EH.secret.keyVault.secretName: eh-conn-string
# Then: Eventstream (source: that Event Hub) -> destination: Eventhouse KQL database.
"""
print(EMITTER_TO_EVENTHOUSE)
print("verification: after one notebook run, .show tables on the target DB should list the "
      "Spark log tables created by the Eventstream mapping.")


# Environment -> Spark properties (session-start scope):
spark.synapse.diagnostic.emitters: EH
spark.synapse.diagnostic.emitter.EH.type: AzureEventHub
spark.synapse.diagnostic.emitter.EH.categories: Log,EventLog,Metrics
spark.synapse.diagnostic.emitter.EH.secret.keyVault: https://<kv>.vault.azure.net/
spark.synapse.diagnostic.emitter.EH.secret.keyVault.secretName: eh-conn-string
# Then: Eventstream (source: that Event Hub) -> destination: Eventhouse KQL database.

verification: after one notebook run, .show tables on the target DB should list the Spark log tables created by the Eventstream mapping.
